# SaoCheck.AI — Modèle d'aide à la vérification

**Projet :** SaoCheck.ai, plateforme de fact-checking assistée par l'IA pour le Tchad et le Sahel
**Cadre :** Code for Africa — AI for Good Fellowship, avec WenakLabs (N'Djamena)
**Auteur :** Pantouin Adjinsala

Ce notebook reproduit, de bout en bout, le modèle d'IA utilisé par la plateforme SaoCheck.ai :

| Brique | Rôle dans la plateforme | Méthode |
|---|---|---|
| 1. Vérifications similaires | Retrouver une rumeur déjà vérifiée (« Déjà vérifié ? ») | Similarité cosinus entre vecteurs de phrases |
| 2. Type de manipulation et thématique | Orienter le vérificateur (affiché à titre indicatif) | Régression logistique sur les vecteurs de phrases |
| 3. Verdict suggéré | Proposé **seulement** s'il atteint des seuils de fiabilité stricts | Régression logistique, texte neutralisé |

Le modèle de phrases est **sentence-CamemBERT** (`dangvantuan/sentence-camembert-base`), utilisé tel quel, sans réentraînement.
Si ce modèle ne peut pas être chargé, le notebook passe automatiquement à une **comparaison lexicale** (fragments de 3 à 5 caractères), comme le fait l'API.

**Principe directeur :** l'IA aide la rédaction, elle ne décide pas. Chaque verdict publié reste une décision humaine.

### Plan
0. Configuration
1. Chargement de la base de référence
2. Exploration
3. Audit des biais : pourquoi on ne prédit pas le verdict directement
4. Représentation des textes (sentence-CamemBERT ou repli lexical)
5. Brique 1 — Vérifications similaires
6. Briques 2 et 3 — Classifieurs évalués par validation croisée groupée
7. Décision de fiabilité
8. Export pour l'API (`classifiers.joblib`, `metrics.json`, `rapport.md`)
9. Essai de prédiction
10. Limites et prochaines étapes

> Le premier essai (CamemBERT affiné sur X-Fact pour prédire le verdict) est documenté dans le notebook `SaoCheck_AI_XFACT_Training.ipynb`. Ses résultats (F1 macro 0,11 en 5 classes, 0,44 en binaire) ont motivé l'approche présentée ici.

## 0. Configuration

**Exécution :** Google Colab ou poste local, Python 3.10 ou plus. Un GPU n'est pas nécessaire : le modèle de phrases n'est pas réentraîné.

**Hors Colab :** `pip install sentence-transformers scikit-learn pandas openpyxl matplotlib joblib`

**Données :** le classeur `Base_de_donnee_SaoCheck.AI.xlsx` (onglet « Base consolidée »). Dans Colab, s'il n'est pas trouvé, une fenêtre d'envoi s'ouvre.

In [ ]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers", "openpyxl"], check=True)

# Classeur consolidé (450 claims) et dossier de sortie
DATA_PATH = os.environ.get("SAOCHECK_DATA", "Base_de_donnee_SaoCheck.AI.xlsx")
OUTPUT_DIR = os.environ.get("SAOCHECK_OUTPUT", "ai_models")

# Modèle de phrases : identique au réglage AI_EMBEDDING_MODEL de l'API. « hashing » force la comparaison lexicale.
ENCODER = os.environ.get("SAOCHECK_ENCODER", "dangvantuan/sentence-camembert-base")

SEED = 0
FOLDS = 5            # validation croisée en 5 parties
MIN_PER_CLASS = 8    # une classe plus rare n'est pas apprise

In [ ]:
import json
import re
import unicodedata
import warnings
from collections import Counter
from datetime import date, datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from openpyxl import load_workbook
from scipy import sparse
from sklearn.feature_extraction.text import HashingVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_recall_fscore_support)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
GREY, DARK, ACCENT = "#9ca3af", "#4b5563", "#af101a"

if IN_COLAB and not Path(DATA_PATH).is_file():
    from google.colab import files
    uploaded = files.upload()
    DATA_PATH = next(iter(uploaded))
print("Classeur :", DATA_PATH)

## 1. Chargement de la base de référence

Le classeur décrit chaque claim en texte libre (une centaine de formulations pour le type d'allégation). On applique **exactement** les correspondances de l'API (`app/ai/labels.py` et `scripts/import_reference_claims.py`) :

- **Type de manipulation** → 7 familles stables (vidéo ou image hors contexte, média manipulé, faux document, arnaque, citation inventée, affirmation ou chiffre) ;
- **Sujet** → thématiques officielles de SaoCheck ; les thèmes seulement *devinés par mots-clés* ne servent pas à l'entraînement ;
- **Verdict** → clés éditoriales SaoCheck. Pour X-Fact, « hors contexte » regroupe en réalité des cas « difficiles à classer » : ils sont écartés.

In [ ]:
MANIPULATION_TYPES = {
    "video_out_of_context": "Vidéo sortie de son contexte",
    "image_out_of_context": "Image sortie de son contexte",
    "manipulated_media": "Média manipulé ou généré par IA",
    "fake_document": "Faux document, communiqué ou compte",
    "scam": "Arnaque ou canular",
    "fabricated_quote": "Citation inventée ou déformée",
    "statement": "Affirmation ou chiffre",
}
VERDICTS = {"faux": "faux", "trompeur / manipule": "trompeur", "hors contexte": "hors-contexte",
            "non verifie": "non-prouve", "vrai": "vrai"}
TOPICS = {
    "politique / gouvernance": "politique-gouvernance", "diplomatie / relations internationales": "politique-gouvernance",
    "defense / securite": "securite", "economie": "economie", "economie / emploi": "economie",
    "infrastructures": "economie", "sante": "sante", "education": "education", "elections": "elections",
    "societe / culture": "societe", "migration / humanitaire": "societe", "sport": "societe",
    "medias / desinformation": "societe", "technologie": "societe", "faits divers": "societe",
    "environnement / ressources": "environnement-climat", "catastrophes / crises": "environnement-climat",
}
THEMATIQUE_LABELS = {
    "politique-gouvernance": "Politique et gouvernance", "sante": "Santé", "economie": "Économie",
    "education": "Éducation", "securite": "Sécurité", "environnement-climat": "Environnement et climat",
    "societe": "Société", "droits-humains": "Droits humains", "conflits": "Conflits",
    "elections": "Élections", "genre": "Genre",
}
# Verdict suggéré : deux groupes seulement, les autres verdicts étant trop rares.
VERDICT_GROUPS = {"faux": "faux_trompeur", "trompeur": "faux_trompeur", "vrai": "vrai"}
VERDICT_GROUP_LABELS = {"faux_trompeur": "Faux ou trompeur", "vrai": "Vrai"}


def fold(value: str) -> str:
    # Minuscules sans accents ni espaces superflus, pour comparer des libellés saisis à la main.
    text = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode()
    return " ".join(text.lower().split())


# Règles appliquées dans l'ordre : la première qui correspond l'emporte.
TYPE_RULES = [
    (r"genere|deepfake|retouche|manipul|une de journal", "manipulated_media"),
    (r"arnaque|canular|hameconnage|offre d.emploi|faux site|fausse offre|recrutement", "scam"),
    (r"citation|propos fabriqu|declaration presidentielle|faux tweet", "fabricated_quote"),
    (r"faux document|communique|faux compte|usurpation|fausse annonce|fausse mesure|faux sondage", "fake_document"),
    (r"video", "video_out_of_context"),
    (r"image|photo|legende", "image_out_of_context"),
    (r"fait|evenement|chiffre|statistique|affirmation|information|rumeur|confirme|nomination|mesure"
     r"|propos|attribution|idee recue|etymologie", "statement"),
]
# Lignes qui décrivent la méthode du fact-checker plutôt qu'une allégation : non étiquetées.
TYPE_IGNORED = re.compile(r"methodologie|auto-correction|clarification")


def manipulation_type(raw):
    if not raw:
        return None
    text = fold(raw)
    if TYPE_IGNORED.search(text):
        return None
    return next((key for pattern, key in TYPE_RULES if re.search(pattern, text)), None)


URL = re.compile(r"https?://\S+|www\.\S+", re.I)
SPACES = re.compile(r"\s+")
# Verbes au conditionnel (« montrerait », « auraient ») : tournure des fact-checkers qui reformulent une rumeur.
CONDITIONAL = re.compile(r"\b\w+(?:rait|raient)\b", re.I)


def clean(text: str) -> str:
    # Texte comparable : sans liens, espaces normalisés, longueur bornée (comme l'API).
    return SPACES.sub(" ", URL.sub(" ", text)).strip()[:2000]


def without_reformulation(text: str) -> str:
    return SPACES.sub(" ", CONDITIONAL.sub(" ", text)).strip()

In [ ]:
SHEET = "Base consolidée"
REQUIRED = ("ID", "Allégation (claim)", "Verdict (harmonisé)")
UNKNOWN = {"", "non précisé", "non renseigné", "nan"}


def _text(value):
    if value is None:
        return None
    text = " ".join(str(value).split())
    return None if text.lower() in UNKNOWN else text


def read_rows(path):
    workbook = load_workbook(path, read_only=True, data_only=True)
    rows = workbook[SHEET].iter_rows(values_only=True)
    header = None
    for row in rows:  # la ligne d'en-tête est celle qui contient « ID »
        cells = [str(cell).strip() if cell is not None else "" for cell in row]
        if "ID" in cells:
            header = cells
            break
    assert header and all(column in header for column in REQUIRED), "En-têtes attendus introuvables"
    records = []
    for row in rows:
        record = dict(zip(header, row))
        if _text(record.get("ID")) and _text(record.get("Allégation (claim)")):
            records.append(record)
    return records


def to_values(record):
    dataset = _text(record.get("Base source"))
    verdict_label = _text(record.get("Verdict (harmonisé)")) or "Non précisé"
    verdict = VERDICTS.get(fold(verdict_label))
    is_xfact = bool(dataset and "x-fact" in dataset.lower())
    if is_xfact and verdict == "hors-contexte":  # « complicated / hard to categorise » dans X-Fact
        verdict, verdict_label = None, "Difficile à classer (X-Fact)"
    kind = manipulation_type(_text(record.get("Type d'allégation")))
    if kind is None and is_xfact:
        kind = "statement"  # X-Fact : déclarations de personnalités et chiffres
    guessed = (_text(record.get("Catégorie devinée (mots-clés)")) or "").lower().startswith("oui")
    topic = _text(record.get("Catégorie de sujet"))
    return {
        "id": str(_text(record["ID"])),
        "claim_text": _text(record.get("Allégation (claim)")),
        "verdict": verdict,
        "verdict_label": verdict_label,
        "manipulation_type": kind,
        "thematique": None if guessed or not topic else TOPICS.get(fold(topic)),
        "platform": _text(record.get("Plateforme")),
        "dataset": dataset,
        "country": _text(record.get("Pays principal")),
        "chad_related": (_text(record.get("Lié directement au Tchad (recalculé)")) or "").lower() == "oui",
        "quality": _text(record.get("Statut qualité")) or "validé",
    }


claims_all = pd.DataFrame([to_values(record) for record in read_rows(DATA_PATH)])
claims_all = claims_all.astype(object).where(claims_all.notna(), None)  # cases vides : None plutôt que NaN
claims = claims_all[claims_all["quality"] == "validé"].reset_index(drop=True)
print(f"{len(claims_all)} claims dans le classeur, dont {len(claims)} validés (utilisés pour l'IA).")
claims.head()

## 2. Exploration

On regarde d'où viennent les claims et comment se répartissent les trois cibles. Le déséquilibre est la première limite du jeu de données : la classe « Affirmation ou chiffre » et les verdicts « Faux » dominent.

In [ ]:
summary = pd.DataFrame({
    "Claims validés": [len(claims)],
    "Liés directement au Tchad": [int(claims["chad_related"].sum())],
    "Avec type de manipulation": [int(claims["manipulation_type"].notna().sum())],
    "Avec thématique fiable": [int(claims["thematique"].notna().sum())],
    "Avec verdict utilisable (vrai / faux / trompeur)": [int(claims["verdict"].map(VERDICT_GROUPS).notna().sum())],
}).T.rename(columns={0: "Nombre"})
display(summary)
display(claims["dataset"].value_counts().rename("Claims par base source").to_frame())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
series = [
    (claims["verdict_label"].value_counts(), "Verdict (harmonisé)"),
    (claims["manipulation_type"].map(MANIPULATION_TYPES).value_counts(), "Type de manipulation"),
    (claims["thematique"].map(THEMATIQUE_LABELS).value_counts(), "Thématique (hors thèmes devinés)"),
]
for ax, (counts, title) in zip(axes, series):
    counts.sort_values().plot.barh(ax=ax, color=GREY, edgecolor=DARK)
    ax.set_title(title, loc="left", fontsize=11, fontweight="bold")
    ax.set_xlabel("claims")
    for i, value in enumerate(counts.sort_values()):
        ax.text(value, i, f" {value}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

## 3. Audit des biais : pourquoi on ne prédit pas le verdict directement

Un modèle qui lit seulement le texte d'une affirmation peut obtenir un bon score **pour de mauvaises raisons**. Trois contrôles :

1. **Le conditionnel.** Les fact-checkers reformulent les rumeurs au conditionnel (« la vidéo *montrerait*… »). Si cette tournure suffit à prédire « faux », le modèle apprend le style de la rédaction, pas la véracité. Or un message réel reçu sur WhatsApp n'est jamais au conditionnel.
2. **Le même modèle sans le conditionnel** : combien de performance reste-t-il ?
3. **Généralisation hors source** : on entraîne sans une base source et on teste sur elle. Si le score s'effondre, le modèle a appris les habitudes d'une source.

In [ ]:
audit = claims.assign(y=claims["verdict"].map(VERDICT_GROUPS)).dropna(subset=["y"]).reset_index(drop=True)
audit["conditionnel"] = audit["claim_text"].str.contains(CONDITIONAL)

rule = np.where(audit["conditionnel"], "faux_trompeur", "vrai")
print(f"Règle « conditionnel → faux ou trompeur » seule : F1 macro {f1_score(audit['y'], rule, average='macro'):.2f}")
display(pd.crosstab(audit["y"].map(VERDICT_GROUP_LABELS), audit["conditionnel"].map({True: "au conditionnel", False: "sans conditionnel"})))


def tfidf_model():
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LogisticRegression(max_iter=2000, class_weight="balanced"))


cv = StratifiedKFold(FOLDS, shuffle=True, random_state=SEED)
raw = cross_val_predict(tfidf_model(), audit["claim_text"], audit["y"], cv=cv)
neutral = cross_val_predict(tfidf_model(), audit["claim_text"].map(without_reformulation), audit["y"], cv=cv)
print(f"TF-IDF, texte brut           : F1 macro {f1_score(audit['y'], raw, average='macro'):.2f}")
print(f"TF-IDF, conditionnel retiré  : F1 macro {f1_score(audit['y'], neutral, average='macro'):.2f}")

In [ ]:
rows = []
for source in audit["dataset"].dropna().unique():
    train, test = audit[audit["dataset"] != source], audit[audit["dataset"] == source]
    if test["y"].nunique() == 0 or train["y"].nunique() < 2:
        continue
    predicted = tfidf_model().fit(train["claim_text"], train["y"]).predict(test["claim_text"])
    counts = test["y"].value_counts()
    rows.append({"Source tenue à l'écart": source, "Claims": len(test),
                 "Exactitude": round(float(np.mean(predicted == test["y"])), 2),
                 "Vrais": int(counts.get("vrai", 0)), "Faux ou trompeurs": int(counts.get("faux_trompeur", 0))})
display(pd.DataFrame(rows).sort_values("Claims", ascending=False))

words = make_pipeline(TfidfVectorizer(min_df=3, sublinear_tf=True),
                      LogisticRegression(max_iter=2000, class_weight="balanced")).fit(audit["claim_text"], audit["y"])
vectorizer, classifier = words.named_steps.values()
names = np.array(vectorizer.get_feature_names_out())
order = np.argsort(classifier.coef_[0])
toward = {classifier.classes_[0]: names[order[:15]], classifier.classes_[1]: names[order[-15:]]}
for label, terms in toward.items():
    print(f"Mots qui tirent vers « {VERDICT_GROUP_LABELS[label]} » : {', '.join(terms)}")

**Lecture.** Si la règle du conditionnel seule obtient déjà un score proche du modèle, si le score baisse quand on retire le conditionnel, si les mots les plus « vrais » ou « faux » sont des noms de sources ou des tournures de rédaction, et si la généralisation hors source est faible, alors un classifieur de verdict apprend des **raccourcis**. C'est ce qu'a montré l'audit : d'où le choix d'une IA d'**aide à la décision**, et d'un verdict suggéré désactivé tant qu'il n'est pas fiable.

## 4. Représentation des textes

- **sentence-CamemBERT** rapproche deux formulations différentes d'une même rumeur (« l'eau de Javel guérit le paludisme » / « boire de la javel soigne le palu »).
- **Repli lexical** : fragments de 3 à 5 caractères, sans modèle à télécharger. Il retrouve surtout les reprises presque mot pour mot. C'est le mode utilisé quand le serveur est trop modeste pour CamemBERT.

Les seuils diffèrent selon la représentation : *rapprochement* (au-delà, la fiche est proposée) et *fort* (badge « Déjà vérifié ? »).

In [ ]:
class HashingEncoder:
    name = "hashing"
    thresholds = {"related": 0.35, "strong": 0.6}

    def __init__(self):
        self._vectorizer = HashingVectorizer(analyzer="char_wb", ngram_range=(3, 5), n_features=2**15,
                                             alternate_sign=False, norm="l2", preprocessor=fold)

    def encode(self, texts):
        matrix = self._vectorizer.transform(texts)
        matrix.data = np.log1p(matrix.data)  # atténue les fragments très répétés
        norms = np.sqrt(np.asarray(matrix.multiply(matrix).sum(axis=1)).ravel())
        norms[norms == 0] = 1
        return sparse.csr_matrix(sparse.diags(1 / norms) @ matrix)


class SentenceEncoder:
    thresholds = {"related": 0.6, "strong": 0.8}

    def __init__(self, model_name):
        from sentence_transformers import SentenceTransformer
        self.name = model_name
        self._model = SentenceTransformer(model_name, device="cpu")

    def encode(self, texts):
        vectors = self._model.encode(list(texts), batch_size=32, normalize_embeddings=True, show_progress_bar=False)
        return np.asarray(vectors, dtype=np.float32)


def load_encoder(name):
    if name == "hashing":
        return HashingEncoder()
    try:
        return SentenceEncoder(name)
    except Exception as error:  # bibliothèque absente, téléchargement impossible, mémoire insuffisante…
        print(f"⚠ Modèle {name} indisponible ({error.__class__.__name__}) : comparaison lexicale utilisée.")
        return HashingEncoder()


encoder = load_encoder(ENCODER)
print("Représentation utilisée :", encoder.name, "| seuils :", encoder.thresholds)


def as_matrix(vectors):
    return vectors.tocsr() if sparse.issparse(vectors) else np.asarray(vectors)


texts = claims["claim_text"].map(clean).tolist()
reference_vectors = as_matrix(encoder.encode(texts))
print("Vecteurs de la base de référence :", reference_vectors.shape)

## 5. Brique 1 — Vérifications similaires

Pour une nouvelle rumeur, on calcule sa similarité cosinus avec toutes les fiches de la base de référence et on renvoie les plus proches au-dessus du seuil de rapprochement. Les vecteurs étant normalisés, la similarité cosinus est un simple produit scalaire.

Pour simuler un message réel, l'exemple reprend un claim tchadien **sans le conditionnel** et en minuscules.

In [ ]:
def similar_claims(text, limit=5, minimum=None):
    minimum = encoder.thresholds["related"] if minimum is None else minimum
    query = as_matrix(encoder.encode([clean(text)]))
    scores = query @ reference_vectors.T
    scores = np.asarray(scores.todense() if sparse.issparse(scores) else scores).ravel()
    best = np.argsort(-scores)[:limit]
    result = claims.loc[best, ["id", "claim_text", "verdict_label", "dataset"]].assign(similarite=scores[best].round(3))
    result["deja_verifie"] = result["similarite"] >= encoder.thresholds["strong"]
    return result[result["similarite"] >= minimum]


rng = np.random.default_rng(SEED)
# Un claim lié au Tchad et reformulé au conditionnel par le fact-checker (sinon, n'importe quel claim)
pool = claims[claims["chad_related"] & claims["claim_text"].str.contains(CONDITIONAL)]
pool = pool if len(pool) else claims
example = pool.iloc[int(rng.integers(len(pool)))]["claim_text"]
message = without_reformulation(example).lower()
print("Message simulé :", message, "\n")
display(similar_claims(message))

In [ ]:
# Essayez avec votre propre texte. minimum=0 affiche les 5 plus proches, même sous le seuil de rapprochement.
similar_claims("Le gouvernement va couper internet samedi dans tout le pays", minimum=0)

## 6. Briques 2 et 3 — Classifieurs

Trois têtes de **régression logistique** (classes rééquilibrées) sont entraînées sur les vecteurs de phrases :

| Tête | Cible | Seuil de fiabilité |
|---|---|---|
| `manipulation` | 7 types de manipulation | F1 macro ≥ 0,50 |
| `thematique` | thématiques officielles | F1 macro ≥ 0,50 |
| `verdict` | « Faux ou trompeur » / « Vrai », texte **sans conditionnel** | F1 macro ≥ 0,75 **et** rappel de chaque classe ≥ 0,60 |

Les seuils du verdict sont volontairement hauts : un verdict suggéré à tort coûte plus cher qu'aucune suggestion.

**Évaluation honnête.** Une même rumeur apparaît parfois sous plusieurs formulations. Si l'une est en entraînement et l'autre en test, le score est gonflé. On regroupe donc les reformulations très proches (similarité au-dessus du seuil « fort ») et la validation croisée en 5 parties ne les sépare jamais (`StratifiedGroupKFold`). La référence de comparaison est un modèle qui prédirait toujours la classe la plus fréquente.

In [ ]:
HEADS = {
    "manipulation": dict(target=lambda row: row["manipulation_type"], labels=MANIPULATION_TYPES,
                         min_macro_f1=0.5, min_recall=0.0, preprocess=None),
    "thematique": dict(target=lambda row: row["thematique"], labels=THEMATIQUE_LABELS,
                       min_macro_f1=0.5, min_recall=0.0, preprocess=None),
    "verdict": dict(target=lambda row: VERDICT_GROUPS.get(row["verdict"] or ""), labels=VERDICT_GROUP_LABELS,
                    min_macro_f1=0.75, min_recall=0.6, preprocess="without_reformulation"),
}


def near_duplicate_groups(features, threshold):
    # Regroupe les reformulations proches (union-find sur la similarité), pour ne pas les séparer.
    similarity = features @ features.T
    similarity = similarity.toarray() if sparse.issparse(similarity) else np.asarray(similarity)
    parent = list(range(similarity.shape[0]))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    rows, cols = np.where(np.triu(similarity, k=1) >= threshold)
    for i, j in zip(rows, cols):
        parent[find(i)] = find(j)
    return np.array([find(i) for i in range(len(parent))])


def make_model(sparse_features):
    classifier = LogisticRegression(max_iter=3000, class_weight="balanced")
    if sparse_features:  # repli lexical : réglage fixe, une recherche serait trop lente
        return classifier.set_params(C=2.0)
    return GridSearchCV(classifier, {"C": [0.25, 1.0, 4.0, 16.0]}, scoring="f1_macro",
                        cv=StratifiedKFold(3, shuffle=True, random_state=SEED))


def train_head(name, spec):
    pairs = [(row, spec["target"](row)) for _, row in claims.iterrows()]
    pairs = [(row, label if isinstance(label, str) else None) for row, label in pairs]  # NaN → pas d'étiquette
    counts = Counter(label for _, label in pairs if label)
    kept = {label for label, count in counts.items() if count >= MIN_PER_CLASS}
    pairs = [(row, label) for row, label in pairs if label in kept]
    if len(kept) < 2:
        print(f"[{name}] ignoré : moins de deux classes avec {MIN_PER_CLASS} exemples.")
        return None

    texts = [clean(row["claim_text"]) for row, _ in pairs]
    if spec["preprocess"] == "without_reformulation":
        texts = [without_reformulation(text) or text for text in texts]
    y = np.array([label for _, label in pairs])
    features = as_matrix(encoder.encode(texts))
    groups = near_duplicate_groups(features, encoder.thresholds["strong"])

    grouped = all(len(set(groups[y == label])) >= FOLDS for label in kept)
    splitter = (StratifiedGroupKFold(FOLDS, shuffle=True, random_state=SEED).split(features, y, groups) if grouped
                else StratifiedKFold(FOLDS, shuffle=True, random_state=SEED).split(features, y))
    predicted = np.empty_like(y)
    for train, test in splitter:
        predicted[test] = make_model(sparse.issparse(features)).fit(features[train], y[train]).predict(features[test])

    labels = sorted(kept)
    precision, recall, f1, support = precision_recall_fscore_support(y, predicted, labels=labels, zero_division=0)
    majority = Counter(y).most_common(1)[0][0]
    metrics = {
        "samples": len(y),
        "classes": {label: int(count) for label, count in zip(labels, support)},
        "macro_f1": round(float(f1_score(y, predicted, average="macro")), 4),
        "accuracy": round(float(accuracy_score(y, predicted)), 4),
        "baseline_macro_f1": round(float(f1_score(y, np.full_like(y, majority), average="macro")), 4),
        "per_class": {label: {"precision": round(float(p), 4), "recall": round(float(r), 4), "f1": round(float(f), 4)}
                      for label, p, r, f in zip(labels, precision, recall, f1)},
        "confusion": {"labels": labels, "matrix": confusion_matrix(y, predicted, labels=labels).tolist()},
        "merged_near_duplicates": int(len(groups) - len(set(groups))),
        "grouped_cv": bool(grouped),
    }
    reliable = metrics["macro_f1"] >= spec["min_macro_f1"] and min(recall) >= spec["min_recall"]

    fitted = make_model(sparse.issparse(features)).fit(features, y)  # modèle final sur toutes les données
    head = {
        "labels": labels,
        "model": getattr(fitted, "best_estimator_", fitted),
        "metrics": metrics,
        "reliable": bool(reliable),
        "thresholds": {"min_macro_f1": spec["min_macro_f1"], "min_recall": spec["min_recall"]},
        "preprocess": spec["preprocess"],
    }
    return head, y, predicted


heads, cv_predictions = {}, {}
for name, spec in HEADS.items():
    result = train_head(name, spec)
    if result:
        heads[name], *cv_predictions[name] = result
        m = heads[name]["metrics"]
        print(f"[{name}] {m['samples']} exemples, {len(heads[name]['labels'])} classes — "
              f"F1 macro {m['macro_f1']:.2f} (classe majoritaire : {m['baseline_macro_f1']:.2f}) — "
              f"{'fiable' if heads[name]['reliable'] else 'NON fiable'}")

In [ ]:
for name, (y_true, y_pred) in cv_predictions.items():
    labels_map = HEADS[name]["labels"]
    labels = heads[name]["labels"]
    display(Markdown(f"#### Tête `{name}` — rapport par classe (validation croisée)"))
    print(classification_report([labels_map[v] for v in y_true], [labels_map[v] for v in y_pred],
                                digits=2, zero_division=0))
    fig, ax = plt.subplots(figsize=(1.1 * len(labels) + 3, 0.8 * len(labels) + 2.5))
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred, labels=labels, display_labels=[labels_map[v] for v in labels],
        cmap="Greys", colorbar=False, ax=ax, xticks_rotation=35)
    ax.set_title(f"Matrice de confusion — {name}", loc="left", fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 7. Décision de fiabilité

Une tête **non fiable** reste affichée comme simple indice (type, thématique), jamais comme un résultat. Le **verdict**, lui, n'est pas proposé du tout tant qu'il n'est pas fiable, même si l'administration l'active.

In [ ]:
decision = pd.DataFrame([{
    "Tête": name,
    "Exemples": head["metrics"]["samples"],
    "Classes": len(head["labels"]),
    "F1 macro": head["metrics"]["macro_f1"],
    "Classe majoritaire seule": head["metrics"]["baseline_macro_f1"],
    "Rappel minimal": min(v["recall"] for v in head["metrics"]["per_class"].values()),
    "Seuil F1": head["thresholds"]["min_macro_f1"],
    "Seuil rappel": head["thresholds"]["min_recall"],
    "Validation groupée": head["metrics"]["grouped_cv"],
    "Décision": "fiable" if head["reliable"] else "non fiable",
} for name, head in heads.items()]).set_index("Tête")
display(decision)

fig, ax = plt.subplots(figsize=(8, 3.2))
x = np.arange(len(decision))
ax.bar(x - 0.2, decision["Classe majoritaire seule"], 0.4, color="#e5e7eb", edgecolor=GREY, label="Classe majoritaire seule")
ax.bar(x + 0.2, decision["F1 macro"], 0.4, color=GREY, edgecolor=DARK, label="Modèle (validation croisée)")
for i, threshold in enumerate(decision["Seuil F1"]):
    ax.hlines(threshold, i - 0.45, i + 0.45, colors=ACCENT, linestyles="--", linewidth=1.5)
ax.set_xticks(x, decision.index)
ax.set_ylim(0, 1)
ax.set_ylabel("F1 macro")
ax.set_title("Scores par tête — en pointillé : seuil de fiabilité", loc="left", fontsize=11, fontweight="bold")
ax.legend(frameon=False, fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

## 8. Export pour l'API

Le fichier `classifiers.joblib` a exactement le format attendu par l'API (`app/ai/classifiers.py`). Il n'est valable **que** pour la représentation avec laquelle il a été entraîné (champ `encoder`) : l'API l'ignore si son réglage `AI_EMBEDDING_MODEL` est différent.

> **Sécurité.** Charger un fichier joblib exécute du code Python. Ne déposez dans `AI_MODELS_DIR` qu'un fichier produit par l'équipe (ce notebook ou `python -m scripts.train_ai_models`), jamais un fichier reçu d'ailleurs. Ne le versionnez pas dans le dépôt : régénérez-le.

In [ ]:
artifact = {
    "version": 1,
    "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "encoder": encoder.name,
    "dataset_size": len(claims),
    "heads": heads,
}
output = Path(OUTPUT_DIR)
output.mkdir(parents=True, exist_ok=True)
joblib.dump(artifact, output / "classifiers.joblib")

summary = {k: v for k, v in artifact.items() if k != "heads"} | {
    "heads": {name: {k: v for k, v in head.items() if k != "model"} for name, head in heads.items()}}
(output / "metrics.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

lines = ["# Classifieurs SaoCheck — rapport d'entraînement", "",
         f"- Date : {artifact['trained_at']}", f"- Modèle de phrases : `{artifact['encoder']}`",
         f"- Claims utilisés : {artifact['dataset_size']} (lignes « validé » de la base de référence)",
         f"- Évaluation : validation croisée en {FOLDS} parties, reformulations proches gardées ensemble", ""]
for name, head in heads.items():
    m = head["metrics"]
    lines += [f"## {name}", "",
              f"- F1 macro : **{m['macro_f1']:.2f}** (classe majoritaire seule : {m['baseline_macro_f1']:.2f})",
              f"- Exactitude : {m['accuracy']:.2f} sur {m['samples']} exemples",
              f"- Décision : **{'fiable' if head['reliable'] else 'non fiable'}** "
              f"(seuils : F1 macro ≥ {head['thresholds']['min_macro_f1']}, rappel minimal ≥ {head['thresholds']['min_recall']})",
              "", "| Classe | Exemples | Précision | Rappel | F1 |", "|---|---:|---:|---:|---:|"]
    lines += [f"| {HEADS[name]['labels'].get(label, label)} | {m['classes'][label]} | {v['precision']:.2f} "
              f"| {v['recall']:.2f} | {v['f1']:.2f} |" for label, v in m["per_class"].items()]
    lines.append("")
(output / "rapport.md").write_text("\n".join(lines) + "\n", encoding="utf-8")

for path in sorted(output.iterdir()):
    print(f"{path}  ({path.stat().st_size / 1024:.0f} Ko)")

## 9. Essai de prédiction

Même logique que l'API : les probabilités de chaque tête, et le verdict seulement si sa tête est fiable.

In [ ]:
def analyse(text):
    cleaned = clean(text)
    plain = encoder.encode([cleaned])
    neutral = encoder.encode([without_reformulation(cleaned) or cleaned])
    out = {}
    for name, head in heads.items():
        if name == "verdict" and not head["reliable"]:
            out[name] = "désactivé (sous le seuil de fiabilité)"
            continue
        features = neutral if head["preprocess"] == "without_reformulation" else plain
        probabilities = head["model"].predict_proba(features)[0]
        ranked = sorted(zip(head["model"].classes_, probabilities), key=lambda pair: -pair[1])[:3]
        out[name] = ", ".join(f"{HEADS[name]['labels'].get(label, label)} ({p:.0%})" for label, p in ranked)
        if not head["reliable"]:
            out[name] += "  — indicatif"
    return pd.Series(out, name=text[:80])


display(pd.concat([
    analyse("Une vidéo montre des soldats tchadiens attaqués à la frontière, elle date en fait de 2015 au Mali"),
    analyse("Recrutement à la douane : envoyez 10 000 FCFA par mobile money pour valider votre dossier"),
    analyse(message),
], axis=1))

## 10. Limites et prochaines étapes

**Limites**
- Jeu de données petit et déséquilibré : 401 claims validés, dont peu de « vrais », et certaines classes rares (citations inventées, médias manipulés).
- Les sorties sont une **aide à la décision** : type et thématique sont indicatifs ; le verdict reste désactivé tant qu'il n'atteint pas les seuils.
- Le texte seul ne contient pas la preuve : la véracité dépend de sources extérieures que le modèle ne consulte pas.
- La comparaison lexicale ne retrouve pas les reformulations ; elle n'est qu'un repli pour les serveurs modestes.

**Prochaines étapes**
1. Collecte ciblée (plan de collecte) : claims vrais du Tchad et du Sahel, types sous-représentés, formulations d'origine et paires de reformulations.
2. Réentraîner et réévaluer avec ce notebook, puis décider du verdict suggéré au regard des seuils.
3. Évaluer la brique 1 sur les paires de reformulations (rappel des fiches déjà vérifiées).
4. Pilote avec la rédaction de SaoCheck et intégration de ses retours.

**Pour l'API** : copier `classifiers.joblib` dans `AI_MODELS_DIR`, puis *Administration → Analyse automatique → Recharger*. Ou, sur le serveur : `python -m scripts.import_reference_claims Base_de_donnee_SaoCheck.AI.xlsx` puis `python -m scripts.train_ai_models`.